In [1]:
import os
from google.colab import drive
import pandas as pd

# 1. Mount Google Drive to keep files permanent across sessions
drive.mount("/content/drive")

# 2. Set path to your existing folder in Google Drive (update if named differently)
PROJECT_DIR = "/content/drive/MyDrive/loan_project"
CSV_PATH = os.path.join(PROJECT_DIR, "data", "loan_train.csv")

# 3. Load your CSV if present; otherwise download the 614-row dataset into your folder
if not os.path.exists(CSV_PATH):
  url = "https://raw.githubusercontent.com/shrikant-temburwar/Loan-Prediction-Dataset/master/train.csv"
  df = pd.read_csv(url)
  df.to_csv(CSV_PATH, index=False)
  print("Downloaded and saved dataset to Google Drive.")
else:
  df = pd.read_csv(CSV_PATH)
  print("Loaded existing dataset from Google Drive.")

# 4. Verify shape and columns
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Mounted at /content/drive
Downloaded and saved dataset to Google Drive.
Shape: (614, 13)
Columns: ['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status']


In [2]:
from sklearn.model_selection import train_test_split

# 1. Drop ID column and encode target ('Y' -> 1, 'N' -> 0)
df_clean = df.drop(columns=["Loan_ID"], errors="ignore").copy()
df_clean["Loan_Status"] = df_clean["Loan_Status"].map({"Y": 1, "N": 0})

# 2. Separate features (X) and target (y)
X = df_clean.drop(columns=["Loan_Status"])
y = df_clean["Loan_Status"]

# 3. Stratified 80/20 Train/Test Split (locks test set away safely)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# 4. Inspect missing values & skewness on TRAINING data only
print(f"Train shape: {X_train.shape} | Test shape: {X_test.shape}")
print(f"Train Approval Rate: {y_train.mean():.3f}\n")
print("Missing values in X_train:")
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])
print("\nNumeric Skewness in X_train:")
print(
    X_train[["ApplicantIncome", "CoapplicantIncome", "LoanAmount"]].skew().round(2)
)

Train shape: (491, 11) | Test shape: (123, 11)
Train Approval Rate: 0.686

Missing values in X_train:
Gender              11
Married              3
Dependents           8
Self_Employed       27
LoanAmount          20
Loan_Amount_Term    12
Credit_History      43
dtype: int64

Numeric Skewness in X_train:
ApplicantIncome      6.55
CoapplicantIncome    7.61
LoanAmount           2.72
dtype: float64


In [3]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

# 1. Define feature groups
skewed_cols = ["ApplicantIncome", "CoapplicantIncome", "LoanAmount"]
num_cols = ["Loan_Amount_Term", "Credit_History"]
cat_cols = [
    "Gender",
    "Married",
    "Dependents",
    "Education",
    "Self_Employed",
    "Property_Area",
]

# 2. Build leak-free preprocessing pipelines
skewed_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scaler", StandardScaler()),
])
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("scaler", StandardScaler()),
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("skewed", skewed_pipe, skewed_cols),
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
])

# 3. Evaluate Rule Baseline (Approve if Credit_History == 1, fill missing with 1)
rule_preds = (X_train["Credit_History"].fillna(1.0) == 1.0).astype(int)
print(
    f"Rule Baseline -> Precision: {precision_score(y_train, rule_preds):.3f} | "
    f"Recall: {recall_score(y_train, rule_preds):.3f} | "
    f"F1: {f1_score(y_train, rule_preds):.3f}"
)

Rule Baseline -> Precision: 0.781 | Recall: 0.982 | F1: 0.870


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

# 1. Define candidate model pipelines
models = {
    "Logistic Regression": Pipeline([
        ("prep", preprocessor),
        ("clf", LogisticRegression(random_state=42, max_iter=1000)),
    ]),
    "Random Forest (Pruned)": Pipeline([
        ("prep", preprocessor),
        (
            "clf",
            RandomForestClassifier(
                n_estimators=100,
                max_depth=5,
                min_samples_leaf=5,
                random_state=42,
            ),
        ),
    ]),
}

# 2. Run 5-Fold Stratified Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["precision", "recall", "f1", "roc_auc"]

for name, pipe in models.items():
  res = cross_validate(
      pipe, X_train, y_train, cv=cv, scoring=scoring, return_train_score=True
  )
  print(f"=== {name} ===")
  print(
      f"Train F1: {res['train_f1'].mean():.3f} | CV F1:"
      f" {res['test_f1'].mean():.3f}"
  )
  print(
      f"CV Precision: {res['test_precision'].mean():.3f} | "
      f"CV Recall: {res['test_recall'].mean():.3f} | "
      f"CV ROC-AUC: {res['test_roc_auc'].mean():.3f}\n"
  )

=== Logistic Regression ===
Train F1: 0.870 | CV F1: 0.870
CV Precision: 0.782 | CV Recall: 0.982 | CV ROC-AUC: 0.730

=== Random Forest (Pruned) ===
Train F1: 0.872 | CV F1: 0.869
CV Precision: 0.780 | CV Recall: 0.982 | CV ROC-AUC: 0.750



In [5]:
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV

# 1. Tune Logistic Regression regularization on Training folds
param_grid = {
    "clf__C": [0.01, 0.1, 0.5, 1.0, 5.0],
    "clf__class_weight": [None, "balanced"],
}

grid = GridSearchCV(
    models["Logistic Regression"],
    param_grid=param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
)
grid.fit(X_train, y_train)
best_model = grid.best_estimator_

# 2. Evaluate ONCE on the untouched Test Set
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print(f"Best Params: {grid.best_params_}")
print(
    f"Test Precision: {precision_score(y_test, y_pred):.3f} | "
    f"Test Recall: {recall_score(y_test, y_pred):.3f}"
)
print(
    f"Test F1-Score:  {f1_score(y_test, y_pred):.3f} | "
    f"Test ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}"
)
print("\nTest Confusion Matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, y_pred))

Best Params: {'clf__C': 0.01, 'clf__class_weight': None}
Test Precision: 0.832 | Test Recall: 0.988
Test F1-Score:  0.903 | Test ROC-AUC: 0.851

Test Confusion Matrix [[TN, FP], [FN, TP]]:
[[21 17]
 [ 1 84]]


In [8]:
import os
import joblib
import pandas as pd

# 1. Save the complete pipeline to Google Drive
MODEL_PATH = os.path.join(PROJECT_DIR, "models", "loan_pipeline.joblib")
joblib.dump(best_model, MODEL_PATH)
print(f"Saved pipeline to: {MODEL_PATH}")

# 2. Reload the pipeline into a fresh variable
loaded_pipeline = joblib.load(MODEL_PATH)

# 3. Create 2 raw edge-case applicants (exactly as a web form sends them)
sample_applicants = pd.DataFrame([
    {
        "Gender": "Male",
        "Married": "Yes",
        "Dependents": "1",
        "Education": "Graduate",
        "Self_Employed": "No",
        "ApplicantIncome": 6000,
        "CoapplicantIncome": 2000.0,
        "LoanAmount": 140.0,
        "Loan_Amount_Term": 360.0,
        "Credit_History": 1.0,
        "Property_Area": "Urban",
    },
    {
        "Gender": "Female",
        "Married": "No",
        "Dependents": "0",
        "Education": "Not Graduate",
        "Self_Employed": "Yes",
        "ApplicantIncome": 2200,
        "CoapplicantIncome": 0.0,
        "LoanAmount": 260.0,
        "Loan_Amount_Term": 360.0,
        "Credit_History": 0.0,
        "Property_Area": "Rural",
    },
])

# 4. Verify predictions match before and after reloading
preds = loaded_pipeline.predict(sample_applicants)
probs = loaded_pipeline.predict_proba(sample_applicants)[:, 1]
for i, (p, prob) in enumerate(zip(preds, probs), 1):
  status = "Approved" if p == 1 else "Rejected"
  print(f"Applicant {i}: {status} (Approval Probability: {prob:.1%})")

Saved pipeline to: /content/drive/MyDrive/loan_project/models/loan_pipeline.joblib
Applicant 1: Approved (Approval Probability: 74.8%)
Applicant 2: Rejected (Approval Probability: 29.5%)


In [9]:
!pip install -q gradio
import os, joblib
import gradio as gr
import pandas as pd

MODEL_PATH = os.path.join(PROJECT_DIR, "models", "loan_pipeline.joblib")
pipeline = joblib.load(MODEL_PATH)


def predict_loan(
    gender,
    married,
    dependents,
    education,
    self_emp,
    app_inc,
    coapp_inc,
    loan_amt,
    term,
    credit_hist,
    prop_area,
):
  if loan_amt <= 0 or app_inc < 0 or coapp_inc < 0:
    return "Invalid Input: Amounts must be positive.", 0.0
  row = pd.DataFrame([{
      "Gender": gender,
      "Married": married,
      "Dependents": dependents,
      "Education": education,
      "Self_Employed": self_emp,
      "ApplicantIncome": app_inc,
      "CoapplicantIncome": coapp_inc,
      "LoanAmount": loan_amt,
      "Loan_Amount_Term": float(term),
      "Credit_History": 1.0 if credit_hist == "1.0 (Good)" else 0.0,
      "Property_Area": prop_area,
  }])
  pred = int(pipeline.predict(row)[0])
  prob = float(pipeline.predict_proba(row)[0, 1])
  status = "✅ Approved" if pred == 1 else "❌ Rejected"
  return status, round(prob * 100, 1)


demo = gr.Interface(
    fn=predict_loan,
    inputs=[
        gr.Dropdown(["Male", "Female"], value="Male", label="Gender"),
        gr.Dropdown(["Yes", "No"], value="Yes", label="Married"),
        gr.Dropdown(["0", "1", "2", "3+"], value="0", label="Dependents"),
        gr.Dropdown(
            ["Graduate", "Not Graduate"], value="Graduate", label="Education"
        ),
        gr.Dropdown(["No", "Yes"], value="No", label="Self Employed"),
        gr.Number(value=5000, label="Applicant Monthly Income ($)"),
        gr.Number(value=1500, label="Co-Applicant Monthly Income ($)"),
        gr.Number(value=130, label="Loan Amount (in $1,000s)"),
        gr.Dropdown(
            [180, 360, 480], value=360, label="Loan Term (Months/Days)"
        ),
        gr.Radio(
            ["1.0 (Good)", "0.0 (Poor)"],
            value="1.0 (Good)",
            label="Credit History",
        ),
        gr.Dropdown(
            ["Urban", "Semiurban", "Rural"],
            value="Semiurban",
            label="Property Area",
        ),
    ],
    outputs=[
        gr.Textbox(label="Loan Decision"),
        gr.Number(label="Approval Probability (%)"),
    ],
    title="Loan Eligibility Prediction Assistant",
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1214851778fc28f006.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [10]:
loan-approval-prediction/
│
├── data/
│   ├── loan_train.csv
│   └── loan_test.csv
│
├── notebooks/
│   └── loan_prediction.ipynb
│
├── models/
│   └── best_model.pkl
│
├── app/
│   └── app.py
│
├── requirements.txt
├── README.md
├── .gitignore
└── LICENSE

SyntaxError: invalid character '│' (U+2502) (1774160351.py, line 2)

In [11]:
pip freeze > requirements.txt

In [ ]:
import os

# 1. Navigate to your project folder in Google Drive
os.chdir("/content/drive/MyDrive/loan_project")

# 2. Initialize Git and set your credentials
!git init
!git config --global user.email "abrehamchru15@gmail.com"
!git config --global user.name "abrishcs"

# 3. Add and commit all project files
!git add .
!git commit -m "Initial commit: Loan approval ML pipeline and Gradio app"

# 4. Connect to your GitHub repository and push
# (Note: Use a GitHub Personal Access Token instead of your password if prompted)
!git branch -M main
!git remote remove origin 2>/dev/null  # Removes old remote if it exists
!git remote add origin https://github.com/YOUR_USERNAME/loan-approval-predictor.git
!git push -u origin main